In [39]:

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder




from sklearn.metrics import accuracy_score

In [40]:

df = pd.read_csv("/content/heart_disease_risk_dataset_earlymed.csv")
df.head()


,Chest_Pain,Shortness_of_Breath,Fatigue,Palpitations,Dizziness,Swelling,Pain_Arms_Jaw_Back,Cold_Sweats_Nausea,High_BP,High_Cholesterol,Diabetes,Smoking,Obesity,Sedentary_Lifestyle,Family_History,Chronic_Stress,Gender,Age,Heart_Risk
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,48.0,0.0
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,46.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,66.0,0.0
3,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,60.0,1.0
4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,69.0,0.0


In [41]:
df.columns

Index(['Chest_Pain', 'Shortness_of_Breath', 'Fatigue', 'Palpitations',
       'Dizziness', 'Swelling', 'Pain_Arms_Jaw_Back', 'Cold_Sweats_Nausea',
       'High_BP', 'High_Cholesterol', 'Diabetes', 'Smoking', 'Obesity',
       'Sedentary_Lifestyle', 'Family_History', 'Chronic_Stress', 'Gender',
       'Age', 'Heart_Risk'],
      dtype='object')

In [42]:
df.isnull().sum()

,0
Chest_Pain,0
Shortness_of_Breath,0
Fatigue,0
Palpitations,0
Dizziness,0
Swelling,0
Pain_Arms_Jaw_Back,0
Cold_Sweats_Nausea,0
High_BP,0
High_Cholesterol,0


In [43]:
df.duplicated().sum()

np.int64(6245)

In [44]:
df["Heart_Risk"].value_counts()

,count
Heart_Risk,
0.0,35000
1.0,35000


In [45]:
X = df.drop("Heart_Risk", axis=1)
y = df["Heart_Risk"]

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [47]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (56000, 18)
X_test: (14000, 18)
y_train: (56000,)
y_test: (14000,)


In [48]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [49]:
baseline_model = DecisionTreeClassifier(random_state=42)

baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

baseline_cv_accuracy = baseline_cv_scores.mean()

In [50]:
baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_test_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline CV Accuracy:", baseline_cv_accuracy)
print("Baseline Test Accuracy:", baseline_test_accuracy)

Baseline CV Accuracy: 0.9806071428571428
Baseline Test Accuracy: 0.9817857142857143


In [51]:
grid_params = {
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [1, 5],
    "criterion": ["gini", "entropy"]
}

In [52]:
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=grid_params,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 5, 7], 'min_samples_leaf': [1, 5],
                         'min_samples_split': [2, 10]},
             scoring='accuracy')

In [53]:
grid_cv_accuracy = grid_search.best_score_

grid_predictions = grid_search.best_estimator_.predict(X_test)

grid_test_accuracy = accuracy_score(
    y_test,
    grid_predictions
)

print("Best Grid Parameters:", grid_search.best_params_)
print("GridSearch CV Accuracy:", grid_cv_accuracy)
print("GridSearch Test Accuracy:", grid_test_accuracy)

Best Grid Parameters: {'criterion': 'gini', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2}
GridSearch CV Accuracy: 0.9676785714285714
GridSearch Test Accuracy: 0.966


In [54]:
random_params = {
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"]
}

In [55]:
random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=random_params,
    n_iter=15,
    cv=cv,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=DecisionTreeClassifier(random_state=42), n_iter=15,
                   n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [3, 5, 7, 10, 15, None],
                                        'min_samples_leaf': [1, 2, 5, 10],
                                        'min_samples_split': [2, 5, 10, 15]},
                   random_state=42, scoring='accuracy')

In [56]:
random_cv_accuracy = random_search.best_score_

random_predictions = random_search.best_estimator_.predict(X_test)

random_test_accuracy = accuracy_score(
    y_test,
    random_predictions
)

print("Best Randomized Parameters:", random_search.best_params_)
print("RandomizedSearch CV Accuracy:", random_cv_accuracy)
print("RandomizedSearch Test Accuracy:", random_test_accuracy)

Best Randomized Parameters: {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'entropy'}
RandomizedSearch CV Accuracy: 0.9813035714285714
RandomizedSearch Test Accuracy: 0.9813571428571428


In [57]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [58]:
rf_cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

rf_cv_accuracy = rf_cv_scores.mean()

In [59]:
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_test_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

print("Random Forest CV Accuracy:", rf_cv_accuracy)
print("Random Forest Test Accuracy:", rf_test_accuracy)

Random Forest CV Accuracy: 0.9915892857142857
Random Forest Test Accuracy: 0.9916428571428572


In [60]:
results = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "Decision Tree - GridSearchCV",
        "Decision Tree - RandomizedSearchCV",
        "Random Forest"
    ],

    "CV Accuracy": [
        baseline_cv_accuracy,
        grid_cv_accuracy,
        random_cv_accuracy,
        rf_cv_accuracy
    ],

    "Test Accuracy": [
        baseline_test_accuracy,
        grid_test_accuracy,
        random_test_accuracy,
        rf_test_accuracy
    ]
})

In [61]:
results["CV Accuracy"] = results["CV Accuracy"].round(4)
results["Test Accuracy"] = results["Test Accuracy"].round(4)

results

,Model,CV Accuracy,Test Accuracy
0,Baseline Decision Tree,0.9806,0.9818
1,Decision Tree - GridSearchCV,0.9677,0.9660
2,Decision Tree - RandomizedSearchCV,0.9813,0.9814
3,Random Forest,0.9916,0.9916


### Conclusion

The models were evaluated using 5-fold cross-validation on the training data and accuracy on an untouched test set. Random Forest performed the best, achieving **99.16% CV accuracy** and **99.16% test accuracy**. Its nearly identical CV and test scores show that it performs consistently on unseen data. Therefore, Random Forest was selected as the best model for predicting heart disease risk.
